In [ ]:
## install openai library
!pip install openai

In [ ]:
## import necessary libraries
import os
from openai import OpenAI
import openai
import json

In [ ]:
## Mount colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Activate API key
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
import json
import pandas as pd

with open("/content/drive/MyDrive/MYPROJECTFOLDER/QSQs.json", "r") as f:
    data = json.load(f)

QSQs_data = pd.DataFrame(
    list(data.items()),
    columns=["Qid", "Q"]
)

QSQs_data.head()

In [ ]:
## let's look at the unique Qids:
QSQs_data.Qid.value_counts()

In [ ]:
## let's look at the unique texts:
QSQs_data.Q.value_counts()

In [ ]:
## let's look at an example of the text
QSQs_data.Q.iloc[65]

In [ ]:
## create function to return API message content
def gpt_response(prompt, input_text):
    completion = client.chat.completions.create(
      model="gpt-4.1-mini",
      messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": input_text}
        ]
      )
    return completion.choices[0].message.content

In [ ]:
# defensive API calling
import time

def gpt_response(prompt, input_text, retries=3, delay=3):
    """
    Robust GPT API request: retries if it fails, always returns a string,
    and logs the error if unsuccessful.
    """
    for attempt in range(1, retries + 1):
        try:
            completion = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": input_text}
                ]
            )
            response_text = completion.choices[0].message.content
            if response_text is None or not response_text.strip():
                raise ValueError("Empty response from API")
            return response_text
        except Exception as e:
            print(f"GPT call failed on attempt {attempt}: {e}")
            time.sleep(delay)  # wait before retrying

    # All retries failed, return a placeholder
    print(f"GPT call ultimately failed after {retries} attempts for input: {input_text[:80]}")
    return ""  # Or you might return "{}" to indicate empty JSON

In [ ]:
## test function on example Q
prompt = """CONTEXT: You will be given a question posed by a Member of Parliament (MP)
    during question time in Zimbabwe's National Assembly (lower house).
    The question, asked between September 2015 and August 2023, is addressed to a government minister.
    ###
    TASK: Extract information from the question to classify it according to the following variables.
    Pay close attention to both explicit and implicit details.
    Always consider the broader context rather than isolated keywords.
    ###
    IMPORTANT CODING RULE:
    For any variable whose instructions specify coding as "NA" unless a condition is met,
    you MUST return "NA" (not 0 or 1) if the required conditions are NOT met.
    ###
    VARIABLES TO CLASSIFY
    1.	nature (Question Type)
        o	Classify whether the MP:
            -	requests a service (e.g., construction, repair, provision), or
            -	inquires about government policy on a specific issue or portfolio area.
        o	IMPORTANT: MPs sometimes phrase service requests as policy questions
            to avoid their question being ruled out by the Speaker.
            Always interpret the question's intent in context.
        o	IMPORTANT: MPs sometimes frame questions as points of order OR privilege
            to immediately get the floor for asking follow-up questions.
            Always interpret the question's intent in context.
        o	If uncertain, code as "DK" with a brief rationale.

        o	These 4 examples are BORDERLINE SERVICE:
             -	EXAMPLE 1: "Hon. Minister, whilst you are addressing the issue of increasing
                            the stock of breed, you have a challenge told by farmers in Chipinge
                            District when you had a visit recently. Their livestock is affected
                            by wildlife at a massive scale. You assured them that you would come back
                            with a response. What is it that you have put in place as Minister responsible
                            for livestock so that the livestock in Chipinge is maintained and the issue of
                            wildlife is managed so we do not continuously experience this problem as a result
                            of the Chiredzi Conservancy?"
                +	EXPLANATION: farmers in Chipinge District expect that the minister solves the issue of livestock loss.
                               = "service"

             -	EXAMPLE 2: "Hon. Minister, I am asking on your concrete plans to mitigate the unfolding
                            water crisis especially in the rural constituencies such as Silobela which
                            are currently confronted with critical water shortages for both human beings
                            and livestock. The situation has been exacerbated by a massive number of boreholes
                            being broken down and no spares available from the DDF."
                +	EXPLANATION: constituents of Silobela expect that the ministry repairs the boreholes.
                               = "service"

             -	EXAMPLE 3: "We see a lot of increase in armed robberies. As a Ministry, what are you doing to
                            protect people, especially at filling stations? When people report to police, they
                            are told there are no vehicles to follow up the crimes. So what are you doing to
                            protect the public from these increased robberies?"
                +	EXPLANATION: request to deliver vehicles so the police can investigate crimes.
                               = "service"

             -	EXAMPLE 4: "What measures are taken at the border post to control the influx of returning
                            residents? Bus loads of people are coming in with COVID certificates and are not
                            tested at Beitbridge Border Post. I am afraid we might end up having
                            the South African variant."
                +	EXPLANATION: the ministry is to test returning residents for COVID at Beitbridge Border Post.
                               = "service"

        o	These 4 examples are BORDERLINE POLICY:
             -	EXAMPLE 1: "Hon. Minister, I want to discuss as to how the issue of bullying in schools
                            can be mitigated. First, teaching kindness and empathy. Second, create opportunities
                            for children to connect with one another. Third, identification of gateway behaviours.
                            Fourth, creating content. Fifth, minimising concentric circles in school. Sixth,
                            participation in simulation fro prevention. Thank you for allowing me to ventilate on
                            issues that the people of Chegutu West Constituency would have me ventilate on."
                +	EXPLANATION: MP sketches policy program that can mitigate bullying at school.
                               = "policy"

              -	EXAMPLE 2: "Hon. Minister, we have recently seen very good collaborative work between the Ministry
                            of Infrastructural Development and the Ministry of Local Government to try and fill up
                            our potholes which were there before the initiative, but which had been made worse by the
                            rains that have fallen. But as a matter of fact, the responsibility to construct, maintain
                            and rehabilitate our local roads lies naturally, if not exclusively, within the purview
                            of our local authorities. Local authorities quite clearly have abdicated their responsibilities.
                            What is the Ministry of Local Government doing to make sure that those local authorities are
                            brought to book, held to account?"
                +	EXPLANATION: MP wants to know the ministry's policy to enforce accountability.
                               = "policy"

              -	EXAMPLE 3: "Hon. Minister, you talked about a good samaritan: a lady who is doing the service in Mbare.
                            Did you take into consideration the issue of sanitation? Are we now resorting to the old system
                            where we are not providing service in the hospital but ana mbuya nyamukuta will deliver babies?
                            Secondly, you are talking about the loan schemes. How are these doctors and nurses going to pay back
                            the loans? Right now, they cannot afford even to buy meals or pay for bus fares. Another issue:
                            must doctors now go and get transport from ZUPCO because, in cases of emergency, a doctor is required
                            to rush to the hospital and save life? When I was young, doctors owned cars to make their work easy."
                +	EXPLANATION: MP wants to know what policy the ministry pursues to improve public healthcare.
                               = "policy"

              -	EXAMPLE 4: "Could the Hon. Minister furnish with reasons as to why ‘it cannot be upgraded to a central
                            hospital’? He gives an impression that it is an impossibility. I just want a specification
                            on that aspect so that we really know that it cannot be upgraded; maybe it is because it is
                            in Midlands or what. I understand that the requirements for an upgrade are not so complicated."
                +	EXPLANATION: MP wants the minister to specify what policy aspects or regulation justifies the refusal.
                               = "policy"

        o	If still uncertain, code as "DK" with a brief rationale.

    2.	transport (Service Related to Transport)
        o	If nature = service, classify whether the request relates broadly
            to transport infrastructure or transport safety.
        o	Code as 1 if yes, otherwise 0.
        o	If nature ≠ service, CODE AS "NA".

    3.	road (Road Infrastructure Request)
        o	If nature = service AND transport = 1, classify if the request involves:
            -	new road construction, restoration of damaged roads, or continuation of interrupted work.
        o	Code as 1 if yes, otherwise 0.
        o	If nature = service BUT transport ≠ 1, CODE AS "NA".
        o	If nature ≠ service, CODE AS "NA".

    4.	education (Service Related to Education)
        o	If nature = service, classify whether the request relates to primary, secondary, or tertiary education.
        o	Code as 1 if yes, otherwise 0.
        o	If nature ≠ service, CODE AS "NA".

    5.	school (School Infrastructure Request)
        o	If nature = service AND education = 1, classify if the request includes:
            -	new school construction, restoration of damaged schools, or resumption of halted work.
        o	Code as 1 if yes, otherwise 0.
        o	If nature = service BUT education ≠ 1, CODE AS "NA".
        o	If nature ≠ service, CODE AS "NA".

    6.	health (Service Related to Health)
        o	If nature = service, classify if the request pertains to health services.
        o	Code as 1 if yes, otherwise 0.
        o	If nature ≠ service, CODE AS "NA".

    7.	clinic (Clinic/Hospital Infrastructure Request)
        o	If nature = service AND health = 1, classify if the request includes:
            -	new clinic/hospital construction, restoration, or resumption of halted work.
        o	Code as 1 if yes, otherwise 0.
        o	If nature = service BUT health ≠ 1, CODE AS "NA".
        o	If nature ≠ service, CODE AS "NA".

    8.	water (Service Related to Water)
        o	If nature = service, classify if the request relates to water resources management or supply.
        o	Code as 1 if yes, otherwise 0.
        o	If nature ≠ service, CODE AS "NA".

    9.	well (Borehole Request)
        o	If nature = service AND water = 1, classify if the request involves:
            -	drilling new boreholes, restoring dilapidated boreholes, or resuming interrupted borehole work.
        o	Code as 1 if yes, otherwise 0.
        o	If nature = service BUT water ≠ 1, CODE AS "NA".
        o	If nature ≠ service, CODE AS "NA".

    ###
    FIRST IMPORTANT CODING RULE:
    If nature = service, these variables MUST always be classified as 0 or 1:
        o	transport
        o	education
        o	health
        o	water

    ###
    SECOND IMPORTANT CODING RULE:
    If you INCORRECTLY use "0" instead of "NA" WHERE A CONDITION REQUIRES "NA",
    this will be counted as an error.

    ###
    THIRD IMPORTANT CODING RULE:
        o	For any variable with "DK" as a possible code:
            -	If you code "DK", ALWAYS append a brief rationale after a comma.
            - If you append a brief rationale, NEVER use quotation marks when referring to some parts of the question.
            -	If you select any other allowed value, output ONLY the value WITHOUT RATIONALE.

    ###
    OUTPUT FORMAT
    Return VALID JSON ONLY with the structure below.
    Replace variable values as per your classification:
    {
    "nature": "service | policy | DK, [brief rationale]",
    "transport": 1 | 0 | "NA",
    "road": 1 | 0 | "NA",
    "education": 1 | 0 | "NA",
    "school": 1 | 0 | "NA",
    "health": 1 | 0 | "NA",
    "clinic": 1 | 0 | "NA",
    "water": 1 | 0 | "NA",
    "well": 1 | 0 | "NA"
    }
    ###
    IMPORTANT: DOUBLE CHECK that ONLY VALID JSON with the structure above is returned!
    ###
    FINAL CHECK: Output a brief rationale ONLY AFTER "DK", and NEVER after values that are NOT "DK".
    """

gpt_response(prompt, QSQs_data.Q.iloc[65])

In [ ]:
## optional -- just to see the loop progress
from tqdm import tqdm
import time

## now run function on all entries of our JSON file and store responses
gpt_responses = []

for message in tqdm(QSQs_data.Q):
  response = gpt_response(prompt, message)
  gpt_responses.append(response)
  time.sleep(2)  # wait 1-2 seconds between calls

In [ ]:
QSQs_data["gpt_response"] = gpt_responses

In [ ]:
# Ultimate fix to have a returning loop that catches all failed Qs (up till a max is reached)

def is_valid_json(s):
    try:
        if not isinstance(s, str):
            return False
        s = s.strip()
        if not s.startswith("{"):
            return False
        import json
        json.loads(s)
        return True
    except Exception:
        return False

def gpt_response(prompt, input_text, retries=3, delay=3):
    # ... (your robust function here)
    completion = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": input_text},
        ]
    )
    return completion.choices[0].message.content or ""

def rerun_missing(df, prompt, response_col="gpt_response", max_loops=5, save_path_base=None):
    """
    Loops through the dataframe rerunning GPT on rows with invalid JSON responses
    until all are valid, or max_loops is reached.
    """
    df = df.copy()
    loop = 0
    while True:
        # Mask of invalid responses
        failed_mask = ~df[response_col].apply(is_valid_json)

        # If none failed, we're done
        num_failed = failed_mask.sum()
        print(f"Rerun round {loop}: {num_failed} failed responses to rerun.")
        if num_failed == 0:
            print("All responses valid JSON.")
            break
        if loop >= max_loops:
            print("Maximum rerun loops reached.")
            break

        # Optionally save failed for inspection
        if save_path_base:
            failed_path = f"{save_path_base}_failed_round{loop}.json"
            df[failed_mask][["Qid", "Q", response_col]].to_json(
                failed_path, orient="records", force_ascii=False, indent=2
            )
            print(f"Saved failed responses for round {loop} to {failed_path}")

        # Extract failed Q's
        failed_qs = df.loc[failed_mask, ["Qid", "Q"]]

        print(f"Rerunning GPT for {len(failed_qs)} failed responses...")
        rerun_gpt_responses = []
        for msg in tqdm(failed_qs["Q"]):
            response = gpt_response(prompt, msg)
            rerun_gpt_responses.append(response)
            time.sleep(2)  # or adjust as needed
        rerun_failed = failed_qs.copy()
        rerun_failed[response_col] = rerun_gpt_responses

        # Update the main dataframe
        rerun_response_map = dict(zip(rerun_failed["Qid"], rerun_failed[response_col]))
        def updated_response(row):
            if is_valid_json(row[response_col]):
                return row[response_col]
            return rerun_response_map.get(row["Qid"], row[response_col])
        df[response_col] = df.apply(updated_response, axis=1)

        loop += 1

    return df


final_df = rerun_missing(
    QSQs_data,
    prompt,
    response_col="gpt_response",
    max_loops=5,  # prevent infinite loops
    save_path_base="/content/drive/MyDrive/MYPROJECTFOLDER/QSQs_failed_reruns"
)

# Save final output!
output_path = "/content/drive/MyDrive/MYPROJECTFOLDER/QSQs_gpt_Nat_final.json"
final_df.to_json(output_path, orient="records", force_ascii=False, indent=2)

In [ ]:
# identify which Qids failed & which are valid
def is_valid_json(s):
    try:
        if not isinstance(s, str):
            return False
        s = s.strip()
        if not s.startswith("{"):
            return False
        json.loads(s)
        return True
    except Exception:
        return False

# Find failed rows by mask
failed_mask = ~QSQs_data["gpt_response"].apply(is_valid_json)

# DataFrame of failed requests
failed_questions = QSQs_data.loc[failed_mask, ["Qid", "Q"]]

# Save for re-run
failed_questions.to_json("/content/drive/MyDrive/MYPROJECTFOLDER/failed_q_Nat.json",
                         orient="records", force_ascii=False, indent=2)

# For display or next loop:
print(f"Number of failed GPT responses: {len(failed_questions)}")
display(failed_questions)

In [ ]:
# rerun the failed calls

# Load failed questions file if needed (or use failed_df from above)
failed_filepath = "/content/drive/MyDrive/MYPROJECTFOLDER/failed_q_Nat.json"
rerun_failed = pd.read_json(failed_filepath)

rerun_gpt_responses = []
for msg in tqdm(rerun_failed["Q"]):
    # Defensive API call, just like in your main loop
    response = gpt_response(prompt, msg)
    rerun_gpt_responses.append(response)
    time.sleep(1)  # To avoid rate limit

rerun_failed["gpt_response"] = rerun_gpt_responses

# (Optional) Save rerun outputs for maximum resilience
rerun_failed.to_json("/content/drive/MyDrive/MYPROJECTFOLDER/failed_q_NatRerun.json",
                     orient="records", force_ascii=False, indent=2)

In [ ]:
# Merge the rerun results back into the main dataframe using Qid

# Prepare mapping of Qid to new response
rerun_response_map = dict(zip(rerun_failed["Qid"], rerun_failed["gpt_response"]))

def fill_rerun(row):
    if is_valid_json(row["gpt_response"]):
        return row["gpt_response"]
    # Otherwise, fill in from rerun mapping (if present)
    return rerun_response_map.get(row["Qid"], row["gpt_response"])

QSQs_data["gpt_response_final"] = QSQs_data.apply(fill_rerun, axis=1)

In [ ]:
# For investigative purposes only (where does it break?):
for idx, response in enumerate(QSQs_data["gpt_response"]):
    try:
        json.loads(response)
    except Exception as e:
        print(f"Bad JSON at index {idx}: {response[:100]}")  # show first 100 chars
        print("Error:", e)
        break

In [ ]:
# Strip markdown code fencing before json.loads

import re

def strip_markdown_fence(text):
    """
    Removes markdown code fencing from a string.
    Accepts strings like '```json\n{...}\n```' or '```\n{...}\n```', returns the {...} as a string.
    """
    if isinstance(text, str):
        # Remove ```json ... ``` or ``` ... ```
        text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text, flags=re.IGNORECASE)
    return text

def safe_json_loads(s):
    try:
        s = strip_markdown_fence(s)
        return json.loads(s)
    except Exception:
        return None  # Or raise, or return {}, etc.

In [ ]:
# For investigative purposes only (where does it break?):
def safe_json_loads(s):
    try:
        return json.loads(s)
    except:
        return None  # or {} or your fallback

QSQs_data["gpt_response_dict"] = QSQs_data["gpt_response"].apply(safe_json_loads)

In [ ]:
# For investigative purposes only (where does it break?):
QSQs_data["gpt_response_dict"] = QSQs_data["gpt_response"].apply(safe_json_loads)

In [ ]:
# Diagnostics

not_json_mask = ~QSQs_data["gpt_response"].apply(is_valid_json)
print(QSQs_data.loc[not_json_mask, ["Qid", "Q", "gpt_response"]].head(10))

In [ ]:
# Turn the gpt_response column into actual dictionnaries (original)
QSQs_data["gpt_response_dict"] = QSQs_data["gpt_response"].apply(json.loads)

In [ ]:
# Expand the classification variables into individual columns
gpt_class_df = QSQs_data["gpt_response_dict"].apply(pd.Series)

In [ ]:
# Concatenate
final_df = pd.concat([QSQs_data[["Qid", "Q"]], gpt_class_df], axis=1)

In [ ]:
final_df

In [ ]:
output_path = "/content/drive/MyDrive/MYPROJECTFOLDER/QSQs_gpt_Nat.json"

final_df.to_json(output_path, orient="records", force_ascii=False, indent=2)
